# Antimicrobial Peptide Activity Prediction (E. coli)
Classifying peptides as active / inactive from sequence-derived features.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import peptides
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")

## Configuration

In [ ]:
CSV_PATH = "dbaasp_ecoli.csv"
SEQ_COL  = "SEQUENCE"   # change to match your file
MIC_COL  = "MIC_µgml"  # already in ug/mL in this dataset
UNIT_COL = "UNIT"        # not present here -> values assumed ug/mL

MIC_THRESHOLD = 32           # ug/mL; active if MIC <= threshold
AA = "ACDEFGHIKLMNPQRSTVWY"
RANDOM_STATE = 42

## Load data

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Columns:", list(df.columns))
print("Shape:", df.shape)
df.head()

In [ ]:
required = {"sequence": SEQ_COL, "MIC": MIC_COL}
missing = [f"{role} -> '{col}'" for role, col in required.items() if col not in df.columns]
if missing:
    raise KeyError(
        "Column(s) not found: " + "; ".join(missing)
        + f"\nAvailable columns: {list(df.columns)}"
        + "\nEdit SEQ_COL / MIC_COL in the Configuration cell to match."
    )
has_units = UNIT_COL in df.columns
print("Unit column present:", has_units)

## Clean & filter

In [ ]:
valid_seq = re.compile(f"^[{AA}]+$")

def first_number(x):
    m = re.search(r"[-+]?\d*\.?\d+", str(x))
    return float(m.group()) if m else np.nan

data = df.copy()
data[SEQ_COL] = data[SEQ_COL].astype(str).str.strip().str.upper()
data = data[data[SEQ_COL].apply(lambda s: bool(valid_seq.match(s)))]

data["MIC_value"] = data[MIC_COL].apply(first_number)
data = data[data["MIC_value"].notna() & (data["MIC_value"] > 0)]
print("Rows with valid sequence and numeric MIC:", len(data))

## Standardise MIC units to µg/mL

In [ ]:
def norm_unit(u):
    u = str(u).strip().lower().replace("\u00b5", "u").replace("\u03bc", "u")
    if u in {"ug/ml", "mg/l", "ug/cm3"}:
        return "ug/mL"
    if u in {"um", "umol/l", "umol/ml"}:
        return "uM"
    return None

data["unit_std"] = data[UNIT_COL].apply(norm_unit) if has_units else "ug/mL"
data["MW"] = data[SEQ_COL].apply(lambda s: peptides.Peptide(s).molecular_weight())

def to_ugml(row):
    if row["unit_std"] == "ug/mL":
        return row["MIC_value"]
    if row["unit_std"] == "uM":
        return row["MIC_value"] * row["MW"] / 1000.0   # uM * g/mol / 1000 = ug/mL
    return np.nan

data["MIC_ugml"] = data.apply(to_ugml, axis=1)
dropped = int(data["MIC_ugml"].isna().sum())
data = data[data["MIC_ugml"].notna()]
print(f"Dropped {dropped} row(s) with missing/unconvertible units.")
print("Remaining rows:", len(data))

## Label & class balance

In [ ]:
data["label"] = (data["MIC_ugml"] <= MIC_THRESHOLD).astype(int)
counts = data["label"].value_counts().sort_index()
print("Class balance (0 = inactive, 1 = active):")
print(counts)
ratio = counts.max() / counts.min()
imbalanced = ratio >= 2
print(f"Imbalance ratio: {ratio:.2f} -> {'imbalanced' if imbalanced else 'balanced'}")

## Features

In [ ]:
def features(seq):
    p = peptides.Peptide(seq)
    f = {
        "length": len(seq),
        "net_charge": p.charge(),
        "hydrophobicity": p.hydrophobicity(),
        "hydrophobic_moment": p.hydrophobic_moment() or 0.0,
        "isoelectric_point": p.isoelectric_point(),
        "aliphatic_index": p.aliphatic_index(),
    }
    for a in AA:
        f[f"pct_{a}"] = 100.0 * seq.count(a) / len(seq)
    return f

X = pd.DataFrame([features(s) for s in data[SEQ_COL]], index=data.index)
y = data["label"]
print("Feature matrix:", X.shape)
X.head()

## Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## Models

In [ ]:
cw = "balanced" if imbalanced else None
models = {
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight=cw, random_state=RANDOM_STATE)),
    "Random Forest": RandomForestClassifier(class_weight=cw, random_state=RANDOM_STATE),
    "Gaussian NB": GaussianNB(),
}

In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    }
    print(f"\n=== {name} ===")
    for metric, value in results[name].items():
        print(f"{metric:10s}: {value:.3f}")
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["inactive", "active"])
    plt.title(name)
    plt.show()

## Model comparison

In [ ]:
scores = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1"]]
scores.round(3)

## Random Forest feature importance

In [ ]:
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7, 4))
sns.barplot(x=imp.values, y=imp.index, color="steelblue")
plt.xlabel("Importance")
plt.title("Top 10 features (Random Forest)")
plt.tight_layout()
plt.show()

## Does arrangement matter? Shuffled-sequence control
Shuffle amino acids within each sequence (composition unchanged) and retrain the best model.

In [ ]:
best_name = scores["F1"].idxmax()
print("Best model by F1:", best_name)

rng = np.random.default_rng(RANDOM_STATE)
def shuffle_seq(seq):
    chars = list(seq)
    rng.shuffle(chars)
    return "".join(chars)

shuffled = data[SEQ_COL].apply(shuffle_seq)
X_shuf = pd.DataFrame([features(s) for s in shuffled], index=data.index)

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_shuf, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
m = clone(models[best_name])
m.fit(Xs_train, ys_train)
shuf_f1 = f1_score(ys_test, m.predict(Xs_test), zero_division=0)
orig_f1 = scores.loc[best_name, "F1"]
print(f"Original F1 : {orig_f1:.3f}")
print(f"Shuffled F1 : {shuf_f1:.3f}")
print(f"Drop        : {orig_f1 - shuf_f1:.3f}")